# 08 - OOP

Classes and instances, inheritance and `super()`, polymorphism, encapsulation, composition, properties, class and static methods, and the dunder methods that let your own types behave like builtins.

This notebook is harder than the ones before it, on purpose. Most of the traps here are not syntax — they are the places where Python's object model does something reasonable that you did not ask for: a mutable class attribute shared by every instance, an assignment that silently shadows instead of updating, a `__eq__` that makes your object unhashable, a forgotten `@property` that is always truthy.

---

**How to use this notebook**

1. Read the question. Write your answer in the code cell beneath it.
2. Run the cell. `check(...)` tells you PASS or FAIL, and shows the failing case.
3. Stuck? Open **Hint** first. Open **Solution** only after a genuine attempt.
4. Call `progress()` any time to see how far you have got.

**Difficulty labels** — questions run easiest to hardest down the notebook.

| | |
|---|---|
| L1 Extremely Easy | one concept, one line |
| L2 Easy | one concept applied, a small edge case |
| L3 Intermediate | 2-3 concepts combined, or a non-obvious edge case |
| L4 Somewhat Difficult | a design choice, correctness needs care |
| L5 Hard | multi-concept, subtle failure modes |


In [ ]:
# Run this cell first.
# Finds the repo root by walking up from wherever Jupyter was started, so this
# works whether you launched from the project root, from notebooks/, or
# installed the package with `pip install -e .`
import pathlib
import sys

if "sprintcheck" not in sys.modules:
    here = pathlib.Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "sprintcheck" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

try:
    from sprintcheck import check, progress, reset
except ModuleNotFoundError:
    raise SystemExit(
        "Could not find the sprintcheck package.\n"
        "Open this notebook from inside the cloned repository, or run\n"
        "    pip install -e .\n"
        "from the project root and restart the kernel."
    )

print("Ready. Solve a question, then run check('Q-001', your_answer).")
print("Call progress() any time to see how far you have got.")


---

## L1 - Extremely Easy

### Q-271 - L1 Extremely Easy - Classes

Define a class `Point` whose `__init__` takes `x` and `y` and stores them as attributes of the instance.

Then write `make_point(x, y)` that builds one and returns the **tuple** `(p.x, p.y)` read back off the instance.

<details><summary>Hint</summary>

`__init__` receives the new object as its first parameter. Storing a value on it means assigning to an attribute of that parameter, not to a bare local name.

</details>

<details><summary>Solution</summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


def make_point(x, y):
    p = Point(x, y)
    return (p.x, p.y)
```

**Why:** `self` is not a keyword — it is just the conventional name for the first parameter, which Python fills in with the instance being built. Writing `x = x` inside `__init__` assigns a local that dies at the end of the call; only `self.x = x` attaches anything to the object.

</details>

In [ ]:
class Point:
    def __init__(self, x, y):
        ...


def make_point(x, y):
    ...

check('Q-271', make_point)

### Q-272 - L1 Extremely Easy - Classes

Define a class `Greeter` whose `__init__` takes a `name`, plus a method `greet()` that returns the string `"Hello, NAME!"`.

Then write `greetings(names)` that builds one `Greeter` per name and returns the **list** of their `greet()` results, in order.

<details><summary>Hint</summary>

A method is a `def` indented inside the `class` body, and its first parameter is the instance. `greet` takes no arguments from the caller — everything it needs is already on the object.

</details>

<details><summary>Solution</summary>

```python
class Greeter:
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"Hello, {self.name}!"


def greetings(names):
    return [Greeter(n).greet() for n in names]
```

**Why:** A method is an ordinary function that lives in the class and takes the instance as its first argument; `g.greet()` is sugar for `Greeter.greet(g)`. Forgetting `self` in the signature is the classic first error, and it shows up as a confusing "takes 0 positional arguments but 1 was given".

</details>

In [ ]:
class Greeter:
    def __init__(self, name):
        ...

    def greet(self):
        ...


def greetings(names):
    ...

check('Q-272', greetings)

### Q-273 - L1 Extremely Easy - Attributes

Define a class `Dog` with a **class attribute** `species` set to `"Canis familiaris"`, and an `__init__` that stores an instance attribute `name`.

Then write `dog_facts(name)` that builds one dog and returns the tuple `(d.name, d.species, Dog.species)`.

<details><summary>Hint</summary>

One of the two attributes is written directly in the class body, outside any method. Read both off the instance anyway — attribute lookup does not stop at the instance.

</details>

<details><summary>Solution</summary>

```python
class Dog:
    species = "Canis familiaris"

    def __init__(self, name):
        self.name = name


def dog_facts(name):
    d = Dog(name)
    return (d.name, d.species, Dog.species)
```

**Why:** Attribute lookup checks the instance first and then the class, which is why `d.species` works even though no `__init__` ever set it. A class attribute is created once when the class body runs, so every instance shares the one object — harmless for a constant string, and a genuine trap for a list (Q-284).

</details>

In [ ]:
class Dog:
    ...

    def __init__(self, name):
        ...


def dog_facts(name):
    ...

check('Q-273', dog_facts)

### Q-274 - L1 Extremely Easy - Classes

Define a class `Counter` whose `__init__` sets `self.count` to `0`, with a method `bump()` that adds one to `self.count`.

Then write `run_counter(n)` that builds one counter, calls `bump()` `n` times, and returns the final `count` as an `int`.

<details><summary>Hint</summary>

The method changes state rather than returning a new value — so it has to write back to the attribute it read, on the same object it was called on.

</details>

<details><summary>Solution</summary>

```python
class Counter:
    def __init__(self):
        self.count = 0

    def bump(self):
        self.count += 1


def run_counter(n):
    c = Counter()
    for _ in range(n):
        c.bump()
    return c.count
```

**Why:** `self.count += 1` reads the attribute, adds one, and stores it back on the instance — that persistence between calls is the whole reason an object exists rather than a plain function. A method that mutates state should usually return `None`, not the new value, so nobody is tempted to chain it.

</details>

In [ ]:
class Counter:
    def __init__(self):
        ...

    def bump(self):
        ...


def run_counter(n):
    ...

check('Q-274', run_counter)

---

## L2 - Easy

### Q-275 - L2 Easy - Inheritance

Define `Animal` with an `__init__` taking `name` and a method `describe()` returning `"NAME is an animal"`. Then define `Dog(Animal)` that adds nothing at all — its body is just `pass`.

Write `dog_facts(name)` returning the tuple `(d.describe(), isinstance(d, Animal), issubclass(Dog, Animal))`.

<details><summary>Hint</summary>

An empty subclass still gets its parent's `__init__` and every method. The two booleans ask about the object and about the class respectively — different questions, different builtins.

</details>

<details><summary>Solution</summary>

```python
class Animal:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"{self.name} is an animal"


class Dog(Animal):
    pass


def dog_facts(name):
    d = Dog(name)
    return (d.describe(), isinstance(d, Animal), issubclass(Dog, Animal))
```

**Why:** Subclassing copies nothing: `Dog` holds a reference to `Animal` and lookup walks up the chain at call time, so a later edit to `Animal.describe` is visible from every `Dog` immediately. `isinstance` asks about an object and `issubclass` about a class — passing an instance to `issubclass` raises `TypeError`, which is the usual mix-up.

</details>

In [ ]:
class Animal:
    def __init__(self, name):
        ...

    def describe(self):
        ...


class Dog(Animal):
    ...


def dog_facts(name):
    ...

check('Q-275', dog_facts)

### Q-276 - L2 Easy - Inheritance

Define `Person` with `__init__(self, name)`. Then define `Employee(Person)` with `__init__(self, name, salary)` that delegates the `name` to the parent with `super().__init__(...)` and stores `salary` itself.

Write `employee_facts(name, salary)` returning `(e.name, e.salary, isinstance(e, Person))`.

<details><summary>Hint</summary>

Defining `__init__` in the subclass replaces the parent's entirely — nothing the parent sets happens unless you ask for it. One call, at the top of the child's `__init__`, restores it.

</details>

<details><summary>Solution</summary>

```python
class Person:
    def __init__(self, name):
        self.name = name


class Employee(Person):
    def __init__(self, name, salary):
        super().__init__(name)
        self.salary = salary


def employee_facts(name, salary):
    e = Employee(name, salary)
    return (e.name, e.salary, isinstance(e, Person))
```

**Why:** An overriding `__init__` does not extend the parent's, it *replaces* it, so forgetting `super().__init__(name)` leaves `self.name` unset and the failure surfaces far away as an `AttributeError`. Prefer `super()` over `Person.__init__(self, name)`: it is the only form that keeps working once a class ends up in a multiple-inheritance chain (Q-313).

</details>

In [ ]:
class Person:
    def __init__(self, name):
        ...


class Employee(Person):
    def __init__(self, name, salary):
        ...


def employee_facts(name, salary):
    ...

check('Q-276', employee_facts)

### Q-277 - L2 Easy - Polymorphism

Define `Animal` with `speak()` returning `"..."`, then `Dog(Animal)` and `Cat(Animal)` that **override** `speak()` to return `"Woof"` and `"Meow"`.

Write `chorus(kinds)` where `kinds` is a list of the strings `"dog"`, `"cat"` or `"animal"`. Build one object per entry and return the **list** of `speak()` results, in order.

<details><summary>Hint</summary>

The calling code should look identical for all three — no `if` on the type once the object exists. A dict mapping the string to the class keeps the dispatch in one place.

</details>

<details><summary>Solution</summary>

```python
class Animal:
    def speak(self):
        return "..."


class Dog(Animal):
    def speak(self):
        return "Woof"


class Cat(Animal):
    def speak(self):
        return "Meow"


def chorus(kinds):
    table = {"dog": Dog, "cat": Cat, "animal": Animal}
    return [table[k]().speak() for k in kinds]
```

**Why:** Polymorphism means the *object* decides which `speak` runs, so one loop serves every animal and a new subclass needs no edit to `chorus`. That is the payoff: `if isinstance(a, Dog): ... elif isinstance(a, Cat): ...` has to be rewritten every time the zoo grows.

</details>

In [ ]:
class Animal:
    def speak(self):
        ...


class Dog(Animal):
    ...


class Cat(Animal):
    ...


def chorus(kinds):
    ...

check('Q-277', chorus)

### Q-278 - L2 Easy - Dunder Methods

Define `User` with `__init__(self, name)` and a `__str__` returning `"User: NAME"`.

Write `label(name)` that builds one and returns the tuple `(str(u), f"{u}")` — both elements are strings.

<details><summary>Hint</summary>

You never call the dunder yourself. Two very ordinary pieces of syntax reach for it on your behalf, and the prompt uses both.

</details>

<details><summary>Solution</summary>

```python
class User:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return f"User: {self.name}"


def label(name):
    u = User(name)
    return (str(u), f"{u}")
```

**Why:** `str()`, `print()` and f-string interpolation all route through `__str__`, so defining it once gives you a readable object everywhere a human will look. `__str__` must return a `str` — returning anything else raises `TypeError` at the call site rather than where the bug lives.

</details>

In [ ]:
class User:
    def __init__(self, name):
        ...

    def __str__(self):
        ...


def label(name):
    ...

check('Q-278', label)

### Q-279 - L2 Easy - Dunder Methods

Define `Point` with `__init__(self, x, y)` and a `__repr__` returning `"Point(1, 2)"` for `Point(1, 2)` — the text that would rebuild the object.

Define **no** `__str__`. Write `show(x, y)` returning the 3-tuple `(repr(p), str(p), repr([p]))`.

<details><summary>Hint</summary>

With no `__str__` defined, `str()` does not fall back to the default object text — it falls back to something you *did* define. And a container never asks its elements for their friendly form.

</details>

<details><summary>Solution</summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"


def show(x, y):
    p = Point(x, y)
    return (repr(p), str(p), repr([p]))
```

**Why:** `__str__` falls back to `__repr__` but never the other way round, which is why `__repr__` is the one to define when you only define one. Containers always use `repr` on their elements, so a list of objects with only `__str__` still prints as `<__main__.Point object at 0x...>` — the single most common reason a debug print says nothing useful.

</details>

In [ ]:
class Point:
    def __init__(self, x, y):
        ...

    def __repr__(self):
        ...


def show(x, y):
    ...

check('Q-279', show)

### Q-280 - L2 Easy - Class & Static Methods

Define `MathUtil` with a `@staticmethod` `add(a, b)` returning `a + b`. It takes no `self` and no `cls`.

Write `use_add(a, b)` returning the tuple `(MathUtil.add(a, b), MathUtil().add(a, b))` — called once on the class and once on an instance.

<details><summary>Hint</summary>

The decorator switches off the automatic first argument. Without it, calling through an instance would pass the instance in as `a`.

</details>

<details><summary>Solution</summary>

```python
class MathUtil:
    @staticmethod
    def add(a, b):
        return a + b


def use_add(a, b):
    return (MathUtil.add(a, b), MathUtil().add(a, b))
```

**Why:** `@staticmethod` is a plain function that happens to live in a class namespace: it gets no instance and no class, so both call sites behave identically. Reach for it when a helper is conceptually part of the class but needs none of its state — and be honest that a module-level function is often the better answer.

</details>

In [ ]:
class MathUtil:
    @staticmethod
    def add(a, b):
        ...


def use_add(a, b):
    ...

check('Q-280', use_add)

### Q-281 - L2 Easy - Class & Static Methods

Define `Date` with `__init__(self, year, month, day)` storing three ints, plus a `@classmethod` `from_string(cls, text)` that parses `"2024-05-17"` and returns a `Date`.

Write `parse_date(text)` that uses the classmethod and returns the tuple `(d.year, d.month, d.day)` — three ints.

<details><summary>Hint</summary>

The first parameter is the class itself, so the last line of `from_string` can build an instance without naming `Date` anywhere. `str.split` on the dash gives you three pieces, still as strings.

</details>

<details><summary>Solution</summary>

```python
class Date:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day

    @classmethod
    def from_string(cls, text):
        year, month, day = text.split("-")
        return cls(int(year), int(month), int(day))


def parse_date(text):
    d = Date.from_string(text)
    return (d.year, d.month, d.day)
```

**Why:** This is the *alternative constructor* pattern: `__init__` takes the canonical form and a classmethod adapts every other input shape to it, so parsing lives next to the type it produces. Using `cls(...)` rather than `Date(...)` is what makes the constructor inherit correctly — a subclass gets its own type back for free (Q-308).

</details>

In [ ]:
class Date:
    def __init__(self, year, month, day):
        ...

    @classmethod
    def from_string(cls, text):
        ...


def parse_date(text):
    ...

check('Q-281', parse_date)

### Q-282 - L2 Easy - Properties

Define `Rectangle` with `__init__(self, width, height)` and an `area` **property** that computes `width * height`.

Write `area_of(width, height)` returning `r.area` — accessed as an attribute, with no parentheses.

<details><summary>Hint</summary>

Decorate the method and the call parentheses disappear at every call site. The value must not be stored in `__init__`, or it goes stale when the sides change.

</details>

<details><summary>Solution</summary>

```python
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height


def area_of(width, height):
    return Rectangle(width, height).area
```

**Why:** `@property` lets a computed value be read like a plain attribute, so a field can later become a computation without breaking a single caller. Computing it on demand also means it can never disagree with `width` and `height`, which a value cached in `__init__` eventually will.

</details>

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        ...

    @property
    def area(self):
        ...


def area_of(width, height):
    ...

check('Q-282', area_of)

### Q-283 - L2 Easy - Composition

Define `Book` with `__init__(self, title, pages)`, and `Shelf` which starts with an empty list of books, has `add(book)`, and has `total_pages()` returning the sum of its books' pages.

A shelf is not a kind of book — it **has** books. Write `shelf_pages(specs)` where `specs` is a list of `(title, pages)` tuples; build a shelf, add a `Book` for each, and return `(number_of_books, total_pages)`.

<details><summary>Hint</summary>

Give `Shelf.__init__` its own empty list. `total_pages` should not reach into `book.pages` counts it has cached — it should ask the list it already holds.

</details>

<details><summary>Solution</summary>

```python
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages


class Shelf:
    def __init__(self):
        self.books = []

    def add(self, book):
        self.books.append(book)

    def total_pages(self):
        return sum(b.pages for b in self.books)


def shelf_pages(specs):
    shelf = Shelf()
    for title, pages in specs:
        shelf.add(Book(title, pages))
    return (len(shelf.books), shelf.total_pages())
```

**Why:** Composition — "has-a" — is the default relationship between objects, and inheritance is the exception you reach for only when a subclass genuinely *is* the parent. Note the empty list is created in `__init__`, once per shelf; writing `books = []` in the class body would give every shelf the same list (Q-284).

</details>

In [ ]:
class Book:
    def __init__(self, title, pages):
        ...


class Shelf:
    def __init__(self):
        ...

    def add(self, book):
        ...

    def total_pages(self):
        ...


def shelf_pages(specs):
    ...

check('Q-283', shelf_pages)

---

## L3 - Intermediate

### Q-284 - L3 Intermediate - Attributes

Without running it, predict the output:

```python
class Basket:
    items = []

    def add(self, x):
        self.items.append(x)

a = Basket()
b = Basket()
a.add("apple")
print(b.items)
print(a.items is b.items)
print(Basket.items)
```

Set `answer` to the three printed lines, e.g. `answer = "[]\nFalse\n[]"`.

<details><summary>Hint</summary>

`self.items.append(...)` never assigns to `self.items` — it looks the attribute up and then mutates whatever it finds. Where does that lookup land when the instance has no `items` of its own?

</details>

<details><summary>Solution</summary>

```python
answer = "['apple']\nTrue\n['apple']"
```

**Why:** The class body runs once, so `items` is a single list shared by every instance, and `append` mutates it in place rather than rebinding anything. This is the most expensive beginner bug in OOP because it stays invisible until the second instance exists: mutable defaults belong in `__init__`, never in the class body.

</details>

In [ ]:
answer = ...

check('Q-284', answer)

### Q-285 - L3 Intermediate - Attributes

Without running it, predict the output:

```python
class Config:
    debug = False

a = Config()
b = Config()
a.debug = True
print(a.debug, b.debug)
print(Config.debug)
print("debug" in a.__dict__, "debug" in b.__dict__)
```

Set `answer` to the three printed lines. Mind the spacing — `print` puts one space between arguments.

<details><summary>Hint</summary>

Assigning through an instance never reaches the class. Compare that with Q-284, where nothing was assigned at all — that difference is the whole question.

</details>

<details><summary>Solution</summary>

```python
answer = "True False\nFalse\nTrue False"
```

**Why:** `a.debug = True` *creates* an instance attribute that shadows the class one for `a` only; `b` and `Config` are untouched, and `a.__dict__` now has a key `b.__dict__` does not. The asymmetry is the trap: reading a class attribute through an instance works, but writing one through an instance silently makes a private copy instead.

</details>

In [ ]:
answer = ...

check('Q-285', answer)

### Q-286 - L3 Intermediate - Dunder Methods

Without running it, predict the output:

```python
class Only:
    def __repr__(self):
        return "Only()"

class Both:
    def __repr__(self):
        return "Both-repr"

    def __str__(self):
        return "Both-str"

print(str(Only()), repr(Only()))
print(str(Both()), repr(Both()))
print([Both()])
```

Set `answer` to the three printed lines.

<details><summary>Hint</summary>

One of the two dunders falls back to the other, and the fallback only runs in one direction. The third line asks a container what it shows for its elements.

</details>

<details><summary>Solution</summary>

```python
answer = "Only() Only()\nBoth-str Both-repr\n[Both-repr]"
```

**Why:** `str(x)` uses `__str__` if there is one and otherwise borrows `__repr__`; there is no fallback the other way. A container's own `__repr__` calls `repr` on every element, which is why a list of objects ignores `__str__` completely — define `__repr__` first, and `__str__` only when the friendly form genuinely differs.

</details>

In [ ]:
answer = ...

check('Q-286', answer)

### Q-287 - L3 Intermediate - Encapsulation

Define `Vault` whose `__init__` sets two attributes: `self._hint` to `"under the mat"` and `self.__secret` (two leading underscores) to `"hunter2"`.

Write `peek(name)` that builds a `Vault` and returns `getattr(vault, name)`. Let it raise whatever `getattr` raises — do not catch anything.

<details><summary>Hint</summary>

One underscore is a convention and changes nothing. Two leading underscores make the compiler rewrite the attribute's name inside the class body — work out what it is rewritten *to*, and one of the test cases falls out.

</details>

<details><summary>Solution</summary>

```python
class Vault:
    def __init__(self):
        self._hint = "under the mat"
        self.__secret = "hunter2"


def peek(name):
    vault = Vault()
    return getattr(vault, name)
```

**Why:** A single underscore is documentation — "internal, do not touch" — enforced by nothing. Two leading underscores trigger *name mangling*: inside the class body `self.__secret` is compiled to `self._Vault__secret`, so the attribute is still perfectly reachable from outside once you know the rule. Mangling exists to stop a subclass from colliding with a private name by accident, not to provide security; Python has no private attributes.

</details>

In [ ]:
class Vault:
    def __init__(self):
        ...


def peek(name):
    ...

check('Q-287', peek)

### Q-288 - L3 Intermediate - Dunder Methods

Define `Point` with `__init__(self, x, y)` and an `__eq__` that returns `True` when both coordinates match. When `other` is not a `Point`, return `NotImplemented` rather than `False`.

Write `compare(a, b)` where `a` and `b` are `(x, y)` tuples. Return the pair `(Point(*a) == Point(*b), Point(*a) == a)` — the second comparison is against the raw tuple.

<details><summary>Hint</summary>

`NotImplemented` is a sentinel, not an exception and not `False`. Returning it tells Python you have no opinion, and it then asks the *other* operand — which decides the second element of every case.

</details>

<details><summary>Solution</summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)


def compare(a, b):
    return (Point(*a) == Point(*b), Point(*a) == a)
```

**Why:** Returning `NotImplemented` gives the right-hand operand its turn, and only when it also declines does Python fall back to identity — which is why a `Point` is never equal to a tuple here, and why the answer is `False` rather than an error. Returning `False` instead would be a lie: it would stop a future type that *does* know how to compare itself to a `Point` from ever being asked.

</details>

In [ ]:
class Point:
    def __init__(self, x, y):
        ...

    def __eq__(self, other):
        ...


def compare(a, b):
    ...

check('Q-288', compare)

### Q-289 - L3 Intermediate - Dunder Methods

Define `Playlist` with `__init__(self, tracks)` storing a list, and a `__len__` returning how many tracks it holds.

Write `describe(tracks)` returning the tuple `(len(p), bool(p))`. Define **no** `__bool__` — the second element is the point of the question.

<details><summary>Hint</summary>

`bool()` looks for `__bool__` first and, failing that, tries one other dunder before giving up and saying `True`. You have just defined that other dunder.

</details>

<details><summary>Solution</summary>

```python
class Playlist:
    def __init__(self, tracks):
        self.tracks = list(tracks)

    def __len__(self):
        return len(self.tracks)


def describe(tracks):
    p = Playlist(tracks)
    return (len(p), bool(p))
```

**Why:** Truthiness asks `__bool__`, falls back to `__len__ != 0`, and defaults to `True` when neither exists — so `__len__` alone makes `if playlist:` mean "has tracks" for free. The default is what bites: an object with neither dunder is always truthy, so `if response:` on a custom wrapper can silently never be false.

</details>

In [ ]:
class Playlist:
    def __init__(self, tracks):
        ...

    def __len__(self):
        ...


def describe(tracks):
    ...

check('Q-289', describe)

### Q-290 - L3 Intermediate - Dunder Methods

Define `Row` with `__init__(self, values)` storing a list, and a `__getitem__(self, index)` that simply hands the index straight through to that list.

Write `pick(values, index)` returning `Row(values)[index]`. One test case passes a `slice` object — do not special-case it.

<details><summary>Hint</summary>

Delegating to the list means negative indices and slices already work, because the list itself knows how to handle them. Whatever went inside the brackets arrives as your `index` parameter, unchanged.

</details>

<details><summary>Solution</summary>

```python
class Row:
    def __init__(self, values):
        self.values = list(values)

    def __getitem__(self, index):
        return self.values[index]


def pick(values, index):
    return Row(values)[index]
```

**Why:** `obj[k]` is exactly `type(obj).__getitem__(obj, k)`, and `obj[1:3]` passes a single `slice(1, 3)` object as that one argument — so delegating to a list inherits negative indexing and slicing with no extra code. Defining `__getitem__` also makes the object iterable via the old protocol, which is why a `for` loop over a `Row` works even without `__iter__`.

</details>

In [ ]:
class Row:
    def __init__(self, values):
        ...

    def __getitem__(self, index):
        ...


def pick(values, index):
    ...

check('Q-290', pick)

### Q-291 - L3 Intermediate - Dunder Methods

Define `Inventory` with `__init__(self, items)` storing a list of strings, and a `__contains__` that answers **case-insensitively**: `"sword"` is in an inventory holding `"Sword"`.

Write `has(items, query)` returning `query in Inventory(items)` — a `bool`.

<details><summary>Hint</summary>

`__contains__` receives the thing being searched for and returns a truthy value. Lowercasing both sides is enough; the interesting part is that `in` is now yours to define however you like.

</details>

<details><summary>Solution</summary>

```python
class Inventory:
    def __init__(self, items):
        self.items = list(items)

    def __contains__(self, item):
        return item.lower() in [i.lower() for i in self.items]


def has(items, query):
    return query in Inventory(items)
```

**Why:** `in` is not hard-wired to equality: with `__contains__` defined, the container decides what membership means, which is how a case-insensitive or range-based container is built. Without it Python falls back to iterating and comparing with `==`, so an object with only `__iter__` still supports `in`, just in linear time and with no say in the rule. Note Python coerces the result to a real `bool` for you.

</details>

In [ ]:
class Inventory:
    def __init__(self, items):
        ...

    def __contains__(self, item):
        ...


def has(items, query):
    ...

check('Q-291', has)

### Q-292 - L3 Intermediate - Dunder Methods

Define `Money` with `__init__(self, cents)` and an `__add__` returning a **new** `Money` holding the combined cents — it must not modify either operand.

Write `add_all(amounts)` where `amounts` is a list of ints. Start from `Money(0)`, fold each amount in with the `+` operator, and return the final `.cents` as an `int`. You may not call `sum()`.

<details><summary>Hint</summary>

Overloading `+` means `a + b` calls `a.__add__(b)`. Returning a fresh object rather than mutating `self` is what keeps `+` behaving the way `+` on ints and strings already does.

</details>

<details><summary>Solution</summary>

```python
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __add__(self, other):
        return Money(self.cents + other.cents)


def add_all(amounts):
    total = Money(0)
    for amount in amounts:
        total = total + Money(amount)
    return total.cents
```

**Why:** `+` is expected to be non-mutating — `a + b` leaves both operands alone and produces a third object — so `__add__` returns a new `Money` instead of doing `self.cents += ...`. Mutating in place is what `__iadd__` is for, and a class that confuses the two makes `total = total + item` quietly corrupt `item`. (`sum()` would work here too, since it starts from `0` — but `0 + Money(...)` would need `__radd__`.)

</details>

In [ ]:
class Money:
    def __init__(self, cents):
        ...

    def __add__(self, other):
        ...


def add_all(amounts):
    ...

check('Q-292', add_all)

### Q-293 - L3 Intermediate - Properties

Define `Rectangle` with a `width` **property** whose setter rejects a negative value by raising `ValueError`, storing the accepted value in `self._width`. `__init__(self, width)` must go through the setter, so a bad value is rejected at construction too.

Write `set_width(start, new)` that builds `Rectangle(start)`, assigns `new` to `.width`, and returns the resulting `.width`. Let the `ValueError` escape.

<details><summary>Hint</summary>

The setter is a second method with the same name, decorated with `@width.setter`. For `__init__` to be validated too, it must assign to the public name, not to the underscore one.

</details>

<details><summary>Solution</summary>

```python
class Rectangle:
    def __init__(self, width):
        self.width = width

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        if value < 0:
            raise ValueError("width must not be negative")
        self._width = value


def set_width(start, new):
    r = Rectangle(start)
    r.width = new
    return r.width
```

**Why:** A property setter is the Java-style getter/setter pair without the ceremony: callers keep writing `r.width = 3`, and validation is added later without touching a single call site — which is exactly why Python code does not write defensive accessors up front. Note `__init__` assigns `self.width`, not `self._width`: assigning the private name directly is the standard way people accidentally bypass their own validation.

</details>

In [ ]:
class Rectangle:
    def __init__(self, width):
        ...

    @property
    def width(self):
        ...

    @width.setter
    def width(self, value):
        ...


def set_width(start, new):
    ...

check('Q-293', set_width)

### Q-294 - L3 Intermediate - Class & Static Methods

Define `Widget` with a class attribute `count = 0`, an `__init__(self, name)` that increments it, a `@classmethod made(cls)` returning the current count, and a `@classmethod reset(cls)` setting it back to `0`.

Write `build_and_count(names)` that resets, builds one `Widget` per name, and returns `Widget.made()` as an `int`.

<details><summary>Hint</summary>

The counter belongs to the class, not to any instance — so `__init__` must not write it through `self`. Think about what Q-285 showed happens if it does.

</details>

<details><summary>Solution</summary>

```python
class Widget:
    count = 0

    def __init__(self, name):
        self.name = name
        Widget.count += 1

    @classmethod
    def made(cls):
        return cls.count

    @classmethod
    def reset(cls):
        cls.count = 0


def build_and_count(names):
    Widget.reset()
    for name in names:
        Widget(name)
    return Widget.made()
```

**Why:** `self.count += 1` would read the class attribute and then *write an instance* one, so the shared counter would stay at zero forever while every widget quietly reported `1` — the exact shadowing from Q-285. Naming the class explicitly (or going through a classmethod) is what keeps the write landing where the read came from.

</details>

In [ ]:
class Widget:
    count = 0

    def __init__(self, name):
        ...

    @classmethod
    def made(cls):
        ...

    @classmethod
    def reset(cls):
        ...


def build_and_count(names):
    ...

check('Q-294', build_and_count)

### Q-295 - L3 Intermediate - Inheritance

Define `Logger` with an empty `self.lines` list and a `log(msg)` that appends `msg`. Define `TimestampLogger(Logger)` that **overrides** `log` to prefix `"[t] "` and then hand the work to the parent with `super()`.

Write `logged(messages)` that builds a `TimestampLogger`, logs each message, and returns its `lines` list.

<details><summary>Hint</summary>

An override does not have to replace the parent's behaviour — it can wrap it. Doing the appending yourself would duplicate the one line the parent already owns.

</details>

<details><summary>Solution</summary>

```python
class Logger:
    def __init__(self):
        self.lines = []

    def log(self, msg):
        self.lines.append(msg)


class TimestampLogger(Logger):
    def log(self, msg):
        super().log("[t] " + msg)


def logged(messages):
    logger = TimestampLogger()
    for message in messages:
        logger.log(message)
    return logger.lines
```

**Why:** Extending rather than replacing is the usual reason to override: the subclass adds its own step and delegates the rest, so a later change to how the parent stores lines needs no edit here. Calling `self.log(...)` instead of `super().log(...)` would recurse forever — the single most common way this pattern is broken.

</details>

In [ ]:
class Logger:
    def __init__(self):
        ...

    def log(self, msg):
        ...


class TimestampLogger(Logger):
    def log(self, msg):
        ...


def logged(messages):
    ...

check('Q-295', logged)

### Q-296 - L3 Intermediate - Polymorphism

Define `Square` with `__init__(self, side)` and `Rect` with `__init__(self, w, h)`. They share no base class — each just has its own `area()` method.

Write `total_area(specs)` where each spec is `("square", side)` or `("rect", w, h)`. Build the objects **with a comprehension**, then return the sum of their areas.

<details><summary>Hint</summary>

Nothing in the summing step should ask what type a shape is — that question is answered once, while building. Two unrelated classes that both offer `area()` are already interchangeable here.

</details>

<details><summary>Solution</summary>

```python
class Square:
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side * self.side


class Rect:
    def __init__(self, w, h):
        self.w = w
        self.h = h

    def area(self):
        return self.w * self.h


def total_area(specs):
    shapes = [Square(s[1]) if s[0] == "square" else Rect(s[1], s[2]) for s in specs]
    return sum(shape.area() for shape in shapes)
```

**Why:** This is *duck typing*: `Square` and `Rect` share no ancestor, yet both are usable anywhere an `area()` is wanted, because Python resolves the method on the object at call time rather than on a declared type. Inheritance is one way to guarantee an interface, not the only way — which is why Python libraries accept "anything with a `.read()`" rather than a specific base class.

</details>

In [ ]:
class Square:
    def __init__(self, side):
        ...

    def area(self):
        ...


class Rect:
    def __init__(self, w, h):
        ...

    def area(self):
        ...


def total_area(specs):
    ...

check('Q-296', total_area)

### Q-297 - L3 Intermediate - Encapsulation

Define `Account` storing `self._balance`, exposing a **read-only** `balance` property (a getter and no setter), and a `deposit(amount)` that adds to the private attribute and returns nothing.

Write `try_account(action, amount)` that builds `Account(100)` and then:

- `"read"` — return `a.balance`;
- `"deposit"` — deposit `amount`, then return `a.balance`;
- `"set"` — assign `amount` to `a.balance` and let whatever happens happen.

<details><summary>Hint</summary>

A property with only a getter is not merely undocumented for writing — the write actively fails. Think about which exception type a refused attribute assignment raises.

</details>

<details><summary>Solution</summary>

```python
class Account:
    def __init__(self, balance):
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        self._balance += amount


def try_account(action, amount):
    a = Account(100)
    if action == "read":
        return a.balance
    if action == "deposit":
        a.deposit(amount)
        return a.balance
    a.balance = amount
    return a.balance
```

**Why:** A getter-only property makes the attribute genuinely read-only from outside: the assignment raises `AttributeError` instead of silently creating a second attribute that shadows nothing useful. That is the real value of encapsulation here — every change to the balance has to go through a method that can enforce the rules, while `_balance` stays a plain attribute for the class's own use.

</details>

In [ ]:
class Account:
    def __init__(self, balance):
        ...

    @property
    def balance(self):
        ...

    def deposit(self, amount):
        ...


def try_account(action, amount):
    ...

check('Q-297', try_account)

---

## L4 - Somewhat Difficult

### Q-298 - L4 Somewhat Difficult - Dunder Methods

Without running it, predict the output:

```python
class Tag:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name

a = Tag("x")
print(a == Tag("x"))
print(Tag.__hash__ is None)
try:
    print({a})
except TypeError:
    print("unhashable")
```

Set `answer` to the three printed lines.

<details><summary>Hint</summary>

Equality and hashing have to agree: two objects that compare equal must hash the same. Python cannot keep that promise on your behalf once you write your own `__eq__`, so it does something drastic instead.

</details>

<details><summary>Solution</summary>

```python
answer = "True\nTrue\nunhashable"
```

**Why:** Defining `__eq__` sets `__hash__` to `None` unless you define it too, because the inherited identity-based hash would put two equal objects in different buckets and break every dict and set. The fix is to define `__hash__` over the same fields (`return hash(self.name)`) — or to accept immutability and let `@dataclass(frozen=True)` generate both for you.

</details>

In [ ]:
answer = ...

check('Q-298', answer)

### Q-299 - L4 Somewhat Difficult - Inheritance

Without running it, predict the output:

```python
class A:
    def who(self):
        return "A"

class B(A):
    def who(self):
        return "B"

class C(A):
    def who(self):
        return "C"

class D(B, C):
    pass

print(D().who())
print([cls.__name__ for cls in D.__mro__])
```

Set `answer` to the two printed lines. Give the list exactly as Python would print it.

<details><summary>Hint</summary>

The search is not depth-first — if it were, `A` would be reached before `C` and `C` could never override anything. A shared ancestor is always visited after every class that inherits from it.

</details>

<details><summary>Solution</summary>

```python
answer = "B\n['D', 'B', 'C', 'A', 'object']"
```

**Why:** Python linearises the hierarchy with C3: local order is preserved and no class appears before one of its subclasses, which puts `A` last even though `B` lists it directly. The practical consequence is that `super()` does not mean "my parent" — it means "the next class in *this object's* MRO", which is why `C` gets a turn at all (Q-313).

</details>

In [ ]:
answer = ...

check('Q-299', answer)

### Q-300 - L4 Somewhat Difficult - Dunder Methods

Without running it, predict the output:

```python
class Point:
    def __init__(self, x):
        self.x = x

    def __eq__(self, other):
        return self.x == other.x

a = Point(1)
b = Point(1)
print(a == b)
print(a is b)
print(a in [b])
print([a].index(b))
```

Set `answer` to the four printed lines.

<details><summary>Hint</summary>

`__eq__` changes what `==` means and nothing else. The last two lines are list operations — look up which comparison the list protocol is documented to use.

</details>

<details><summary>Solution</summary>

```python
answer = "True\nFalse\nTrue\n0"
```

**Why:** `is` asks whether two names point at the same object and can never be overridden, so a custom `__eq__` leaves it untouched — two equal points are still two objects. Container searches like `in`, `.index()` and `.remove()` use identity-or-equality (`x is e or x == e`), which is why they follow your `__eq__` while `is` does not.

</details>

In [ ]:
answer = ...

check('Q-300', answer)

### Q-301 - L4 Somewhat Difficult - Properties

Without running it, predict the output. Note the missing decorator:

```python
class Box:
    def __init__(self, n):
        self._n = n

    def size(self):   # oops - no @property
        return self._n

b = Box(0)
print(type(b.size).__name__)
print(b.size == 0)
print(bool(b.size))
print(b.size() == 0)
```

Set `answer` to the four printed lines.

<details><summary>Hint</summary>

Without the decorator, `b.size` never runs the body — it produces the callable itself. Then ask what an arbitrary object with no `__bool__` and no `__len__` is worth in a boolean context.

</details>

<details><summary>Solution</summary>

```python
answer = "method\nFalse\nTrue\nTrue"
```

**Why:** A forgotten `@property` turns every read into a bound method object, and the danger is that nothing raises: the method object is not equal to anything you compare it with, and it is always truthy, so `if box.size:` quietly takes the wrong branch forever. This is why `if user.is_admin:` on a method rather than a property is a genuine security bug — it is always true.

</details>

In [ ]:
answer = ...

check('Q-301', answer)

### Q-302 - L4 Somewhat Difficult - Dunder Methods

Define `Card` with `__init__(self, name, rank)` and a `__lt__` comparing `rank` only.

Write `sort_names(pairs)` where `pairs` is a list of `(name, rank)` tuples. Build the cards, sort them with the plain builtin `sorted` — no `key=` — and return the **list of names** in the resulting order.

<details><summary>Hint</summary>

`sorted` needs exactly one comparison to do its job, and `__lt__` is the one it asks for. Equal ranks should keep their original order; check whether you need to arrange that yourself.

</details>

<details><summary>Solution</summary>

```python
class Card:
    def __init__(self, name, rank):
        self.name = name
        self.rank = rank

    def __lt__(self, other):
        return self.rank < other.rank


def sort_names(pairs):
    cards = [Card(name, rank) for name, rank in pairs]
    return [card.name for card in sorted(cards)]
```

**Why:** `sorted`, `min` and `max` are all built on `<` alone, so a single `__lt__` makes a class orderable — but only for `<`, since Python derives no other operator from it (use `functools.total_ordering`, as in Q-314, for the rest). Python's sort is stable, so ties keep their input order without any effort.

</details>

In [ ]:
class Card:
    def __init__(self, name, rank):
        ...

    def __lt__(self, other):
        ...


def sort_names(pairs):
    ...

check('Q-302', sort_names)

### Q-303 - L4 Somewhat Difficult - Dunder Methods

Define `Multiplier` with `__init__(self, factor)` and a `__call__(self, x)` returning `x * factor`, so the instance can be used like a function.

Write `apply_factor(factor, values)` returning the tuple `(callable(m), [m(v) for v in values])` — a bool and a list.

<details><summary>Hint</summary>

`m(3)` is not special syntax for functions; it is a dunder like any other. The builtin in the first element of the tuple asks whether the *type* defines it.

</details>

<details><summary>Solution</summary>

```python
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):
        return x * self.factor


def apply_factor(factor, values):
    m = Multiplier(factor)
    return (callable(m), [m(v) for v in values])
```

**Why:** Anything with `__call__` is callable, which is what lets a configured object be passed straight to `map`, `sorted(key=...)` or a framework that expects a function. Compared with a closure it gives you the same behaviour plus inspectable state and a useful `__repr__` — that is exactly how decorators with arguments and most "policy object" designs are built.

</details>

In [ ]:
class Multiplier:
    def __init__(self, factor):
        ...

    def __call__(self, x):
        ...


def apply_factor(factor, values):
    ...

check('Q-303', apply_factor)

### Q-304 - L4 Somewhat Difficult - Dunder Methods

Define `Countdown` with `__init__(self, start)` and an `__iter__` that **yields** `start, start - 1, ... , 1`. Write `__iter__` as a generator — no separate iterator class, no `__next__`.

Write `countdown_list(start)` returning `list(Countdown(start))`. For `start = 0` the list is empty.

<details><summary>Hint</summary>

A `def` containing `yield` returns a generator when called — and a generator is already an iterator. That is precisely what `__iter__` is required to hand back.

</details>

<details><summary>Solution</summary>

```python
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        for n in range(self.start, 0, -1):
            yield n


def countdown_list(start):
    return list(Countdown(start))
```

**Why:** Making `__iter__` a generator collapses the whole iterator protocol into one method: Python calls it, gets a fresh generator, and that generator supplies `__next__` and `StopIteration` for you. Because each call produces a *new* generator, the object can be iterated repeatedly — whereas returning `self` from `__iter__` gives a one-shot object that is silently empty the second time round.

</details>

In [ ]:
class Countdown:
    def __init__(self, start):
        ...

    def __iter__(self):
        ...


def countdown_list(start):
    ...

check('Q-304', countdown_list)

### Q-305 - L4 Somewhat Difficult - Inheritance

Using `abc`, define `Shape(ABC)` with an `@abstractmethod name(self)`, then `Circle(Shape)` and `Square(Shape)` returning `"circle"` and `"square"`.

Write `make(kind)`: for `"circle"` and `"square"` return the matching object's `name()`; for anything else, attempt to instantiate `Shape()` itself and let the failure escape.

<details><summary>Hint</summary>

An abstract base class is not blocked from being subclassed — it is blocked from being *instantiated* while any abstract method is unimplemented. That check happens at construction, and it is not an `AttributeError`.

</details>

<details><summary>Solution</summary>

```python
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def name(self):
        ...


class Circle(Shape):
    def name(self):
        return "circle"


class Square(Shape):
    def name(self):
        return "square"


def make(kind):
    if kind == "circle":
        return Circle().name()
    if kind == "square":
        return Square().name()
    return Shape().name()
```

**Why:** `ABCMeta` refuses to construct any class that still has unimplemented abstract methods, so the mistake is caught at the moment of creation rather than when the missing method is finally called in production. That is the trade: an ABC states the interface up front, where duck typing (Q-296) leaves it implicit and fails later.

</details>

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def name(self):
        ...


class Circle(Shape):
    ...


class Square(Shape):
    ...


def make(kind):
    ...

check('Q-305', make)

### Q-306 - L4 Somewhat Difficult - Composition

Define `Engine` with `__init__(self, hp)` and `start()` returning `"vroom"`. Define `Car` which **builds its own** `Engine` in `__init__`, delegates `start()` to it, and exposes the engine's `hp` through a property.

`Car` must not inherit from `Engine`. Write `car_report(make, hp)` returning `(c.make, c.hp, c.start(), isinstance(c, Engine))`.

<details><summary>Hint</summary>

Delegation is a one-line method that forwards the call. The property exists so callers never have to write `car.engine.hp` — the engine stays an implementation detail.

</details>

<details><summary>Solution</summary>

```python
class Engine:
    def __init__(self, hp):
        self.hp = hp

    def start(self):
        return "vroom"


class Car:
    def __init__(self, make, hp):
        self.make = make
        self.engine = Engine(hp)

    @property
    def hp(self):
        return self.engine.hp

    def start(self):
        return self.engine.start()


def car_report(make, hp):
    c = Car(make, hp)
    return (c.make, c.hp, c.start(), isinstance(c, Engine))
```

**Why:** Inheriting would have made a car *be* an engine — true for `isinstance`, false for anyone reading the code, and it would drag in every future `Engine` method whether it made sense on a car or not. Composition exposes only what you forward, so the engine can be swapped for an `ElectricMotor` without a single caller noticing.

</details>

In [ ]:
class Engine:
    def __init__(self, hp):
        ...

    def start(self):
        ...


class Car:
    def __init__(self, make, hp):
        ...

    @property
    def hp(self):
        ...

    def start(self):
        ...


def car_report(make, hp):
    ...

check('Q-306', car_report)

### Q-307 - L4 Somewhat Difficult - Properties

Define `Temp` storing only `self._celsius`, with a `fahrenheit` property that converts on read (`c * 9 / 5 + 32`) and converts back on write (`(f - 32) * 5 / 9`). Store nothing in Fahrenheit.

Write `round_trip(celsius, new_f)` that builds `Temp(celsius)`, records `t.fahrenheit`, then assigns `new_f` to `t.fahrenheit`, and returns `(recorded_fahrenheit, t._celsius)` — two floats.

<details><summary>Hint</summary>

There is one source of truth and one derived view of it. The setter's job is to run the conversion backwards so the single stored field stays authoritative.

</details>

<details><summary>Solution</summary>

```python
class Temp:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self._celsius = (value - 32) * 5 / 9


def round_trip(celsius, new_f):
    t = Temp(celsius)
    recorded = t.fahrenheit
    t.fahrenheit = new_f
    return (recorded, t._celsius)
```

**Why:** A property pair turns a derived value into a writable view of one stored field, so the two units can never drift apart — storing both and "keeping them in sync" is the bug this design removes. Note the results are floats even for whole degrees, because `/` is true division; exact round-trips are not guaranteed either, since `(c * 9 / 5 + 32 - 32) * 5 / 9` is binary floating point, not algebra.

</details>

In [ ]:
class Temp:
    def __init__(self, celsius):
        ...

    @property
    def fahrenheit(self):
        ...

    @fahrenheit.setter
    def fahrenheit(self, value):
        ...


def round_trip(celsius, new_f):
    ...

check('Q-307', round_trip)

### Q-308 - L4 Somewhat Difficult - Class & Static Methods

Define `Base` with `__init__(self, label)` and a `@classmethod create(cls, label)` that returns a new instance. Define `Child(Base)` with an empty body.

Write `created_name(which, label)`: call `create` on `Base` when `which` is `"base"` and on `Child` otherwise, then return `(type(obj).__name__, obj.label)`.

<details><summary>Hint</summary>

`cls` is whichever class the call was made through, not the class the method was written in. Hard-coding the class name in `create` is what breaks the `Child` case.

</details>

<details><summary>Solution</summary>

```python
class Base:
    def __init__(self, label):
        self.label = label

    @classmethod
    def create(cls, label):
        return cls(label)


class Child(Base):
    pass


def created_name(which, label):
    cls = Base if which == "base" else Child
    obj = cls.create(label)
    return (type(obj).__name__, obj.label)
```

**Why:** A classmethod receives the class it was *called on*, so `cls(label)` builds a `Child` when invoked as `Child.create(...)` — the factory is inherited and still returns the right type. Writing `return Base(label)` would work until the first subclass appeared and then silently hand back the wrong class, which is why alternative constructors always use `cls`.

</details>

In [ ]:
class Base:
    def __init__(self, label):
        ...

    @classmethod
    def create(cls, label):
        ...


class Child(Base):
    ...


def created_name(which, label):
    ...

check('Q-308', created_name)

---

## L5 - Hard

### Q-309 - L5 Hard - Dunder Methods

Define `Vector` with `__init__(self, x, y)` and these dunders: `__add__`, `__mul__` (by a number), `__rmul__`, `__eq__` (only against another `Vector`), and `__repr__` giving `"Vector(1, 2)"`.

Write `vector_ops(a, b, k)` where `a` and `b` are `(x, y)` tuples. Return the 4-tuple `(repr(va + vb), repr(va * k), repr(k * va), (va + vb) == (vb + va))`.

<details><summary>Hint</summary>

`k * va` does not call `__mul__` — the int is on the left and gets asked first. Work out what Python does when the left operand declines, and which second dunder that reaches.

</details>

<details><summary>Solution</summary>

```python
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, k):
        return Vector(self.x * k, self.y * k)

    def __rmul__(self, k):
        return self * k

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"


def vector_ops(a, b, k):
    va = Vector(*a)
    vb = Vector(*b)
    return (repr(va + vb), repr(va * k), repr(k * va), (va + vb) == (vb + va))
```

**Why:** For `k * va` Python asks `int.__mul__(k, va)` first, gets `NotImplemented`, and only then tries the *reflected* operation `va.__rmul__(k)` — without it, `2 * v` raises `TypeError` while `v * 2` works, an asymmetry that looks inexplicable from the call site. Every binary operator has this reflected twin, which is also why returning `NotImplemented` rather than `False` from `__eq__` (Q-288) matters.

</details>

In [ ]:
class Vector:
    def __init__(self, x, y):
        ...

    def __add__(self, other):
        ...

    def __mul__(self, k):
        ...

    def __rmul__(self, k):
        ...

    def __eq__(self, other):
        ...

    def __repr__(self):
        ...


def vector_ops(a, b, k):
    ...

check('Q-309', vector_ops)

### Q-310 - L5 Hard - Composition

Define one class `Node` with `__init__(self, name, size=0, children=())` and a `total_size()` returning its own size plus the total size of every child, however deep.

Write `tree_size(spec)` where a spec is either `("file", name, size)` or `("dir", name, [child_specs])`. It must **call itself** for each child. Return the total size as a number. A directory of its own contributes `0`.

<details><summary>Hint</summary>

A node that holds nodes is the composite pattern — `total_size` asks each child the same question it was asked. Collapsing each child subtree into a single sized `Node` keeps the recursion to one line.

</details>

<details><summary>Solution</summary>

```python
class Node:
    def __init__(self, name, size=0, children=()):
        self.name = name
        self.size = size
        self.children = list(children)

    def total_size(self):
        return self.size + sum(child.total_size() for child in self.children)


def tree_size(spec):
    kind, name, payload = spec
    if kind == "file":
        return Node(name, size=payload).total_size()
    children = [Node(c[1], size=tree_size(c)) for c in payload]
    return Node(name, children=children).total_size()
```

**Why:** The composite pattern lets a leaf and a container answer the same question, so `total_size` never asks what kind of node it is holding — the recursion terminates by itself when a node has no children. Note `children=()` as the default rather than `[]`: a mutable default is evaluated once at definition time and shared by every call, which is the function-level form of the class attribute bug in Q-284.

</details>

In [ ]:
class Node:
    def __init__(self, name, size=0, children=()):
        ...

    def total_size(self):
        ...


def tree_size(spec):
    ...

check('Q-310', tree_size)

### Q-311 - L5 Hard - Encapsulation

Define `Frozen` that sets `self.x` and `self.y` in `__init__`, then sets `self._locked = True` as its last statement. Override `__setattr__` so that once `_locked` is set, any further assignment raises `AttributeError`; before that, assignments go through normally.

Write `touch(action, value)` building `Frozen(1, 2)` and then:

- `"read"` — return `(f.x, f.y)`;
- `"write"` — do `f.x = value` and return `(f.x, f.y)`;
- `"delete"` — do `del f.x` and return `hasattr(f, "x")`.

<details><summary>Hint</summary>

`self._locked = True` goes through your own `__setattr__` too, so the guard has to tolerate the flag not existing yet. To actually store a value once you have overridden the hook, call the base implementation explicitly. And ask yourself which dunder `del` uses.

</details>

<details><summary>Solution</summary>

```python
class Frozen:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self._locked = True

    def __setattr__(self, name, value):
        if getattr(self, "_locked", False):
            raise AttributeError(f"{type(self).__name__} is immutable")
        object.__setattr__(self, name, value)


def touch(action, value):
    f = Frozen(1, 2)
    if action == "read":
        return (f.x, f.y)
    if action == "delete":
        del f.x
        return hasattr(f, "x")
    f.x = value
    return (f.x, f.y)
```

**Why:** `__setattr__` intercepts *every* assignment, including the ones inside `__init__`, so it must both tolerate the half-built object and store through `object.__setattr__` — writing `self.name = value` inside it recurses until the stack overflows. The deletion case is the lesson: guarding one hook leaves `__delattr__` wide open, which is why real immutability comes from `@dataclass(frozen=True)`, a `NamedTuple`, or `__slots__` plus discipline rather than from a hand-rolled guard.

</details>

In [ ]:
class Frozen:
    def __init__(self, x, y):
        ...

    def __setattr__(self, name, value):
        ...


def touch(action, value):
    ...

check('Q-311', touch)

### Q-312 - L5 Hard - Dunder Methods

Define two context managers over a shared `log` list. `Tracker` appends `"enter"` in `__enter__` (returning `self`) and `"exit"` in `__exit__`, returning a falsy value. `Swallower` is identical except its `__exit__` returns `True`.

Write `run(kind, should_raise)`: build a `Swallower` for `"swallow"` and a `Tracker` otherwise; in a **`with` block** append `"body"` and raise `ValueError` if asked; after the block append `"after"`; wrap the whole thing in `try/except ValueError` appending `"caught"`. Return the log.

<details><summary>Hint</summary>

`__exit__` receives the exception and its return value decides the exception's fate. One of these two managers stops the `ValueError` from ever reaching your `except` clause — and that changes which of the two trailing entries appears.

</details>

<details><summary>Solution</summary>

```python
class Tracker:
    def __init__(self, log):
        self.log = log

    def __enter__(self):
        self.log.append("enter")
        return self

    def __exit__(self, exc_type, exc, tb):
        self.log.append("exit")
        return False


class Swallower(Tracker):
    def __exit__(self, exc_type, exc, tb):
        super().__exit__(exc_type, exc, tb)
        return True


def run(kind, should_raise):
    log = []
    manager = Swallower(log) if kind == "swallow" else Tracker(log)
    try:
        with manager:
            log.append("body")
            if should_raise:
                raise ValueError("boom")
        log.append("after")
    except ValueError:
        log.append("caught")
    return log
```

**Why:** `__exit__` always runs, and a **truthy** return value suppresses the exception — so `Swallower` makes the `with` statement complete normally and execution continues to `"after"`, while `Tracker` lets the `ValueError` propagate to the `except`. Returning `True` unconditionally is almost always wrong: it hides every error, including ones you never anticipated, which is why the default should be to return `None` and say nothing.

</details>

In [ ]:
class Tracker:
    def __init__(self, log):
        ...

    def __enter__(self):
        ...

    def __exit__(self, exc_type, exc, tb):
        ...


class Swallower(Tracker):
    def __exit__(self, exc_type, exc, tb):
        ...


def run(kind, should_raise):
    ...

check('Q-312', run)

### Q-313 - L5 Hard - Inheritance

Define `Base` with `steps()` returning `["base"]`. Define `Loud(Base)` and `Fast(Base)` whose `steps()` each return their own name prepended to `super().steps()`. Define `Both(Loud, Fast)` doing the same with `"both"`.

Write `pipeline(which)` taking `"both"`, `"loud"`, `"fast"` or `"base"`, building that class and returning its `steps()` list. Every class must use `super()` — never name a parent directly.

<details><summary>Hint</summary>

Inside `Loud.steps`, `super()` does not necessarily mean `Base`. It means the next class along the MRO of the object that is actually running — and for a `Both` instance that is a class `Loud` has never heard of. Q-299 has the ordering rule.

</details>

<details><summary>Solution</summary>

```python
class Base:
    def steps(self):
        return ["base"]


class Loud(Base):
    def steps(self):
        return ["loud"] + super().steps()


class Fast(Base):
    def steps(self):
        return ["fast"] + super().steps()


class Both(Loud, Fast):
    def steps(self):
        return ["both"] + super().steps()


def pipeline(which):
    table = {"both": Both, "loud": Loud, "fast": Fast, "base": Base}
    return table[which]().steps()
```

**Why:** `super()` walks the MRO of `type(self)`, so inside a `Both` instance `Loud.steps`'s `super()` resolves to `Fast` — every class in the chain gets a turn, which is what makes cooperative mixins work. Writing `Base.steps(self)` instead would skip `Fast` entirely and, in a diamond, call the shared ancestor twice; that is the concrete reason `super()` is not just a nicer spelling.

</details>

In [ ]:
class Base:
    def steps(self):
        ...


class Loud(Base):
    def steps(self):
        ...


class Fast(Base):
    def steps(self):
        ...


class Both(Loud, Fast):
    def steps(self):
        ...


def pipeline(which):
    ...

check('Q-313', pipeline)

### Q-314 - L5 Hard - Dunder Methods

Define `Version`, decorated with `functools.total_ordering`. `__init__` takes a string like `"1.2.0"` and stores `self.parts` as a tuple of ints. Define `__eq__` and `__lt__` over `parts` and nothing else — no `__le__`, `__gt__` or `__ge__`.

Write `compare_versions(a, b)` returning the 5-tuple `(va < vb, va <= vb, va > vb, va >= vb, va == vb)` — five bools.

<details><summary>Hint</summary>

Comparing the tuples of ints does all the ordering work, including `"1.10"` beating `"1.2"`. The decorator fills in the three operators you did not write, from the two you did.

</details>

<details><summary>Solution</summary>

```python
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, text):
        self.parts = tuple(int(p) for p in text.split("."))

    def __eq__(self, other):
        return self.parts == other.parts

    def __lt__(self, other):
        return self.parts < other.parts


def compare_versions(a, b):
    va = Version(a)
    vb = Version(b)
    return (va < vb, va <= vb, va > vb, va >= vb, va == vb)
```

**Why:** `@total_ordering` derives `<=`, `>` and `>=` from `__eq__` plus one ordering operator, so a single source of truth cannot drift out of sync the way four hand-written comparisons eventually do. Comparing tuples of ints is also what makes version ordering correct: `"1.10"` sorts *after* `"1.2"` numerically but before it as a string, which is the classic release-numbering bug.

</details>

In [ ]:
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, text):
        ...

    def __eq__(self, other):
        ...

    def __lt__(self, other):
        ...


def compare_versions(a, b):
    ...

check('Q-314', compare_versions)

---

## Mini-project P-08 - Bank account hierarchy

Build a small account hierarchy, then drive it with `run_account(spec, ops)`.

`Account(owner, balance)` stores `self._balance`, exposes a **read-only** `balance` property, and has `deposit(amount)` and `withdraw(amount)`. Both raise `ValueError` for a non-positive amount, and `withdraw` raises `ValueError` when the amount exceeds what is available. Give it a `__repr__` of the form `Account('Ada', 120)` using the **actual class name**, and an `__eq__` that requires the same class, owner and balance.

`Savings(Account)` adds `rate` (default `0.02`) and `add_interest()`, which adds `round(balance * rate, 2)` to the balance. `Current(Account)` adds `overdraft` (default `0`) and may be withdrawn down to `-overdraft` — change *only* what 'available' means, not `withdraw` itself.

`run_account(spec, ops)`: `spec` is `(kind, owner, opening)` with `kind` in `"basic"`/`"savings"`/`"current"` (a current account gets an overdraft of `100`). Each op is `("deposit", n)`, `("withdraw", n)`, `("interest",)` or `("set_balance", n)`. Apply them in order, catching `ValueError` and `AttributeError` and appending the exception's type name to an `errors` list. Return

```python
(repr(acct), acct.balance, errors, acct == Account(owner, acct.balance))
```

<details><summary>Hint</summary>

Put the overdraft rule behind a small overridable hook — a `_available()` method the base class uses in `withdraw` — so the subclass changes one line and inherits every validation. `type(self).__name__` inside `__repr__` gives each subclass the right name for free.

</details>

<details><summary>Solution</summary>

```python
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    def _available(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("withdrawal must be positive")
        if amount > self._available():
            raise ValueError("insufficient funds")
        self._balance -= amount

    def __repr__(self):
        return f"{type(self).__name__}({self.owner!r}, {self._balance})"

    def __eq__(self, other):
        if not isinstance(other, Account):
            return NotImplemented
        return (type(self), self.owner, self._balance) == (
            type(other), other.owner, other._balance,
        )


class Savings(Account):
    def __init__(self, owner, balance=0, rate=0.02):
        super().__init__(owner, balance)
        self.rate = rate

    def add_interest(self):
        self._balance += round(self._balance * self.rate, 2)


class Current(Account):
    def __init__(self, owner, balance=0, overdraft=0):
        super().__init__(owner, balance)
        self.overdraft = overdraft

    def _available(self):
        return self._balance + self.overdraft


def run_account(spec, ops):
    kind, owner, opening = spec
    if kind == "savings":
        acct = Savings(owner, opening)
    elif kind == "current":
        acct = Current(owner, opening, overdraft=100)
    else:
        acct = Account(owner, opening)
    errors = []
    for op in ops:
        try:
            if op[0] == "deposit":
                acct.deposit(op[1])
            elif op[0] == "withdraw":
                acct.withdraw(op[1])
            elif op[0] == "interest":
                acct.add_interest()
            elif op[0] == "set_balance":
                acct.balance = op[1]
        except (ValueError, AttributeError) as exc:
            errors.append(type(exc).__name__)
    return (repr(acct), acct.balance, errors, acct == Account(owner, acct.balance))
```

**Why:** The design move worth keeping is the `_available()` hook: `withdraw` owns every rule that is the same for all accounts, and `Current` overrides the one line that genuinely differs — no copied validation, no `isinstance` check. Two behaviours fall out of the rest: `acct.balance = n` raises `AttributeError` because the property has no setter, and `Savings` is never `==` a plain `Account` because the comparison includes the exact type, so a subclass cannot sneak past an equality check.

</details>

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        ...


class Savings(Account):
    ...


class Current(Account):
    ...


def run_account(spec, ops):
    ...

check('P-08', run_account)

---

## Your progress

In [ ]:
progress()